# Bayesian HUD Statistics

## Section 0 — Introduction

A poker HUD tracks opponent statistics accumulated over thousands of *your* hands played in
total, but the sample sizes for any individual villain are much smaller.  Conditional stats
like 3-bet% (3B%) are rarer still — they are only triggered in roughly 15% of hands, so
after 50 total hands a villain has had only 7–8 three-bet opportunities.

The value of an observed frequency depends on its **signal-to-noise ratio**: how often the
action occurs (the opportunity rate), and how much true variation exists across players (the
prior spread $\sigma$).  When sample noise dominates prior spread, the raw frequency is
unreliable.

Three use cases address this problem at increasing levels of complexity:

| | Parameter space | Data availability | Reliability |
|---|---|---|---|
| Use case 1: Single-stat filtering | Minimal | High | High |
| Use case 2: Archetype-based filtering | Moderate | Moderate | Moderate |
| Use case 3: Within-hand updating | Large | Low | Illustrative |

> **Note on use case 3:** The within-hand updater activates whenever aggregate stats are
> unavailable *or* unreliable — this covers both anonymous zone-poker tables (where no hand
> history accumulates) and thin-sample situations on regular tables.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from bayesian_hud.archetypes import (
    ARCHETYPES, ARCHETYPE_NAMES, STAT_NAMES, MIXTURE_WEIGHTS, get_archetype_params
)
from bayesian_hud.single_stat import (
    plot_estimation_comparison, plot_shrinkage_curves
)
from bayesian_hud.multi_stat import (
    plot_archetype_posteriors, plot_estimate_improvement,
    simulate_archetype_population,
    plot_population_scatter, plot_variance_decomposition,
    plot_correlation_structure, plot_sequential_updating,
)
from bayesian_hud.decision_tree import (
    plot_posterior_evolution, plot_path_tree, trace_path
)

## Section 1 — Use Case 1: Single-Stat Bayesian Filtering

### Bayesian derivation

We model each player's true rate $\theta$ for a given stat as drawn from a population prior:

$$\theta \sim \mathcal{N}(\mu,\; \sigma^2)$$

After observing $n$ opportunities and $k$ successes, the raw sample frequency is
$\hat\theta = k/n$ with sampling noise:

$$\hat s = \sqrt{\frac{\hat\theta\,(1 - \hat\theta)}{n}}$$

Under a Gaussian–Gaussian conjugate model the posterior mean is exactly the
**shrinkage estimator**:

$$\theta_B = \mu + w\,(\hat\theta - \mu), \qquad w = \frac{\sigma}{\sigma + \hat s}$$

This is the proper Bayesian posterior mean — not an ad hoc correction.  The weight
$w \in [0,1]$ balances data against the prior.  When $n$ is small, $\hat s \gg \sigma$
so $w \approx 0$ and the estimate shrinks toward $\mu$.  As $n \to \infty$, $\hat s \to 0$
so $w \to 1$ and the estimate converges to $\hat\theta$.

### Signal-to-noise and the crossover point

The crossover point $n^*$ is the number of opportunities at which the data and the prior
receive equal weight ($w = 0.5$):

$$n^* = \frac{\mu(1-\mu)}{\sigma^2}$$

Below $n^*$ the prior dominates; above it the data dominates.  For rare stats like 3B%,
even 50 total hands yield only ~8 opportunities — almost certainly below $n^*$.

### Population parameters

The parameters $\mu$ and $\sigma$ here are **population-level** quantities — in practice
estimated from your own database across all villains.  In this notebook we derive them as
the mixture-weighted averages over the three archetypes defined in Section 2.

In [ ]:
fig = plot_shrinkage_curves()
plt.show()

In [ ]:
fig = plot_estimation_comparison(total_hands=50)
plt.show()

## Section 2 — Archetype Definitions

The player pool is not homogeneous — it consists of distinct player types with different
tendencies.  We model this as a **mixture of $K = 3$ archetypes**, each carrying a Gaussian
prior over the three stats:

$$\theta_j \mid k \;\sim\; \mathcal{N}\!\left(\mu_{kj},\; \sigma_{kj}^2\right),
\quad j \in \{\text{VPIP, PFR, 3B\%}\}$$

The three archetypes are:

- **Fish** — loose-passive calling station.  High VPIP, very low PFR and 3B%.
- **TAG** — tight-aggressive regular.  Selective preflop range, decent aggression.
- **LAG** — loose-aggressive.  Wide range, high aggression, frequent 3-bets.

The mixture weights $\pi_k$ represent population shares and serve as the prior $P(k)$
before any data is observed.

In [ ]:
mu, sigma, pi = get_archetype_params()

header = f"{'Archetype':<8}  {'Weight':>6}  " + "  ".join(f"{s:>12}" for s in STAT_NAMES)
sep    = "-" * len(header)
print(header)
print(sep)
for k, name in enumerate(ARCHETYPE_NAMES):
    stat_cols = "  ".join(
        f"{mu[k,j]:.2f} +/- {sigma[k,j]:.2f}" for j in range(len(STAT_NAMES))
    )
    print(f"{name:<8}  {pi[k]:>6.0%}  {stat_cols}")
print(sep)
print(f"{'Total':<8}  {pi.sum():>6.0%}")

In [ ]:
print("Mixture weights:")
for name, w in MIXTURE_WEIGHTS.items():
    print(f"  {name}: {w:.0%}")

> **Important caveat:** these parameters are illustrative and hand-specified.  The
> framework is valid regardless of the specific values chosen; in production they should
> be calibrated from your own HUD database via expectation-maximisation or variational
> inference on a Gaussian mixture model.

## Section 3 — Use Case 2: Archetype-Based Multi-Stat Filtering

Rather than a single population prior, we maintain a **discrete posterior over archetypes**
updated jointly on all three stats.

### Model

The total variance of $\hat\theta_j$ under archetype $k$ accounts for both the prior spread
and the sampling noise:

$$v_{kj} = \sigma_{kj}^2 + \hat s_j^2, \qquad \hat s_j = \sqrt{\frac{\hat\theta_j(1-\hat\theta_j)}{n_j}}$$

The log-likelihood of the observed vector $\hat{\boldsymbol\theta}$ under archetype $k$ is:

$$\log p(\hat{\boldsymbol\theta} \mid k) = \sum_j \log \mathcal{N}\!\left(\hat\theta_j;\; \mu_{kj},\; v_{kj}\right)$$

and the posterior is:

$$P(k \mid \hat{\boldsymbol\theta}) \propto \pi_k \cdot p(\hat{\boldsymbol\theta} \mid k)$$

The final estimate for stat $j$ is the archetype-posterior-weighted shrinkage:

$$\hat\theta^{\text{arch}}_j = \sum_k P(k \mid \hat{\boldsymbol\theta})\cdot \hat\theta^B_{kj}$$

where $\hat\theta^B_{kj}$ is the per-archetype shrinkage estimate.

### 3a — The mixture structure

The figure below plots VPIP vs PFR for 800 simulated players (200 hands each), coloured by
true archetype, with $\pm 1\sigma$ Gaussian ellipses overlaid.

In [ ]:
fig = plot_population_scatter()
plt.show()

The global distribution is **multimodal** — three visible clusters in VPIP/PFR space.
A single Gaussian prior would conflate all three player types into one blob, ignoring the
cluster structure entirely.  The archetype model exploits these clusters to produce sharper
per-player estimates.

### 3b — Intra- vs inter-archetype variation

For an archetype to be a useful discriminator, the **between-archetype variance** must be
large relative to the **within-archetype variance**.  By the law of total variance:

$$\text{Var}(\theta_j) = \underbrace{\sum_k \pi_k\,(\mu_{kj} - \bar\mu_j)^2}_{\text{between}}
+ \underbrace{\sum_k \pi_k\,\sigma_{kj}^2}_{\text{within}}$$

where $\bar\mu_j = \sum_k \pi_k \mu_{kj}$ is the population mean for stat $j$.

In [ ]:
fig = plot_variance_decomposition()
plt.show()

Stats with a high between/(between + within) ratio discriminate well between archetypes —
a single observation of that stat shifts the archetype posterior substantially.  Stats with
a low ratio are dominated by within-archetype noise and provide weaker archetype signal.

### 3c — Covariance structure

The figure below shows the correlation matrix of true rates $\boldsymbol\theta$ globally
(left panel) and within each archetype separately (panels 2–4).

In [ ]:
fig = plot_correlation_structure()
plt.show()

**Globally**, VPIP, PFR, and 3B% are strongly correlated because different archetypes
have different joint profiles (Fish: high VPIP, low PFR, low 3B%; LAG: high VPIP, high PFR,
high 3B%).

**Within archetypes**, correlations are much lower — each archetype's stats are approximately
independent by construction in this model.

Multi-stat updating exploits the global covariance: observing high VPIP shifts the archetype
posterior toward Fish/LAG, which immediately adjusts our prior expectation for PFR and 3B%
*even before observing those stats directly*.

### 3d — Sequential updating: how each stat sharpens the posterior

We examine three representative villains and show how the archetype posterior evolves as
each stat is revealed in sequence: VPIP only, then VPIP + PFR, then all three.

In [ ]:
fig = plot_sequential_updating()
plt.show()

- **Fish-like** ($\hat\theta = [0.42, 0.11, 0.02]$): high VPIP already shifts probability
  strongly toward Fish; adding low PFR and low 3B% confirms it.
- **Ambiguous** ($\hat\theta = [0.28, 0.20, 0.07]$): moderate stats consistent with both
  TAG and LAG.  The posterior stays diffuse even after all three stats are observed.
  This is genuine uncertainty — the player's numbers sit in the overlap region.
- **LAG-like** ($\hat\theta = [0.36, 0.27, 0.11]$): moderate VPIP combined with high PFR
  and high 3B% converges toward LAG as more stats are revealed.

The ambiguous case is an important reminder: **when a player's stats straddle archetype
boundaries, the posterior uncertainty is genuine and should be retained** rather than forced
into a single label.

### 3e — Estimation improvement

RMSE comparison across raw frequency, single-stat Bayesian (population prior), and
archetype-based Bayesian — evaluated at 50 hands per player.

In [ ]:
fig = plot_estimate_improvement(total_hands=50)
plt.show()

The archetype model improves most for **Fish** — the most clearly separated cluster —
and least for TAG/LAG whose clusters overlap in VPIP/PFR space.  For 3B%, all methods
struggle at 50 hands (~8 opportunities); archetype Bayes still wins by leveraging the
archetype posterior formed from the more abundant VPIP and PFR observations.

## Section 4 — Use Case 3: Within-Hand Updating (Anonymous / Thin Samples)

### Motivation

Zone poker tables are anonymous — no hand history accumulates across sessions.  More
generally, use case 3 activates whenever aggregate stats are *too thin to be reliable*:
a villain with only 10 hands provides essentially no useful VPIP/PFR signal.

In both cases, the observable space shifts from 3 aggregate stats to **within-hand
sequential actions**, which is a much richer signal set.

### The challenge

This comes at a cost.  The parameter space is now large — each archetype must specify
action probabilities at every decision node.  And each node deep in the tree is reached
only after a specific sequence of earlier actions, so each observation is rare.  Both
problems compound as the tree grows.

**Treat use case 3 as a proof of concept:** the structure is correct, but reliably
calibrating the action probabilities requires a large database and careful estimation.

### Update rule

At each decision node $t$ we observe action $a_t$.  The posterior update is:

$$P(k \mid a_{1:t}) \;\propto\; P(k \mid a_{1:t-1}) \cdot P(a_t \mid k)$$

This is a simple likelihood-weighted rescaling applied at each step.

In [ ]:
fig = plot_path_tree()
plt.show()

In [ ]:
fig = plot_posterior_evolution()
plt.show()

## Section 5 — Custom Hand Trace (Use Case 3)

`trace_path()` accepts any sequence of `(node, action)` tuples and returns the full
posterior history.  We examine two contrasting lines below:

- **Path 1:** preflop call, flop check, vs-cbet call, turn check, vs-barrel raise —
  a passive then suddenly aggressive line.
- **Path 2:** preflop 3-bet — a single early action that provides strong archetype signal.

In [ ]:
paths = {
    "Path 1: call → check → vs-cbet call → check → vs-barrel raise": [
        ("preflop",        "call"),
        ("flop_donk",      "check"),
        ("flop_vs_cbet",   "call"),
        ("turn_donk",      "check"),
        ("turn_vs_barrel", "raise"),
    ],
    "Path 2: preflop 3-bet": [
        ("preflop", "threbet"),
    ],
}

for path_name, seq in paths.items():
    history, final = trace_path(seq)
    print(f"\n{'='*70}")
    print(f"  {path_name}")
    print(f"{'='*70}")
    header = (
        f"  {'Step':<4}  {'Node':<18}  {'Action':<12}  "
        + "  ".join(f"P({n})" for n in ARCHETYPE_NAMES)
    )
    print(header)
    print(f"  {'-'*68}")
    row_labels = [("prior", "-")] + list(seq)
    for i, (node, action) in enumerate(row_labels):
        probs = "  ".join(f"{history[i][k]:>7.3f}" for k in range(len(ARCHETYPE_NAMES)))
        print(f"  {i:<4}  {node:<18}  {action:<12}  {probs}")

## Section 6 — Summary and Extensions

### What the framework achieves

Across all three use cases, Bayesian shrinkage consistently outperforms raw sample averages
in the small-sample regime — exactly where HUD data is least reliable and most consequential
for in-game decisions.  The archetype layer adds a second benefit: by pooling information
across stats, it can identify player type even when each individual stat is too noisy to be
conclusive on its own.  The within-hand updater extends this to anonymous opponents where
no history exists at all.

### Limitations

**Independence assumption.** The model treats VPIP, PFR, and 3B% as conditionally
independent given the archetype.  In reality they are correlated even within archetypes.
This underestimates posterior confidence but rarely changes which archetype has the highest
probability.

**Gaussian approximation.** True rates are bounded in $[0, 1]$, but the Gaussian prior has
unbounded support.  For extreme stats (e.g. 3B% near 0) a Beta prior would be more
faithful; the Gaussian is a convenient approximation that works well away from the
boundaries.

**Fixed archetype parameters.** The $\mu_{kj}$, $\sigma_{kj}$, and $\pi_k$ values are
hand-specified here.  In production they should be estimated from a large labelled HUD
database via expectation-maximisation or variational inference on a Gaussian mixture model.

**Action probabilities.** The within-hand likelihoods in `archetypes.py` are heuristic.
Calibrating them from real hand histories — stratified by position and board texture — is
the most impactful improvement available.

### Natural extensions

- **Correlated stats:** replace independent Gaussians with a multivariate Gaussian prior
  $\boldsymbol\theta \mid k \sim \mathcal{N}(\boldsymbol\mu_k, \boldsymbol\Sigma_k)$.
  The covariance $\boldsymbol\Sigma_k$ can be estimated from data or set via elicitation.
  Multi-stat updating then exploits cross-stat correlations explicitly.
- **Online updating:** as hands accumulate session by session, update the posterior
  incrementally — the Bayesian framework supports this natively since today's posterior is
  tomorrow's prior.
- **Estimating archetype parameters from HUD databases:** use EM on a Gaussian mixture
  model, stratified by stake and player pool, to learn $\mu_{kj}$, $\sigma_{kj}$, and
  $\pi_k$ directly from data.
- **Expanding the decision tree:** condition action probabilities on board texture (dry vs
  wet), pot odds, or villain positional tendencies to sharpen within-hand inference.